# Import necessary dependencies and settings

In [ ]:
# importa pandas y numpy
import pandas as pd
import numpy as np


# Transforming Nominal Features

Nominal attributes consist of discrete categorical values with no notion or sense of order amongst them. The idea here is to transform these attributes into a more representative numerical format which can be easily understood by downstream code and pipelines. Let?s look at a new dataset pertaining to video game sales.

In [ ]:
# lee 'vgsales.csv'
# muestra las primeras 6 filas de las columnas 'Name', 'Platform', 'Year', 'Genre', 'Publisher'

vg_df = pd.read_csv("Ficheros/vgsales.csv", encoding = "utf-8")
vg_df[['Name', 'Platform', "Year", "Genre", "Publisher"]].iloc[0:7]

### Get the list of unique video game genres 

In [ ]:
generos = vg_df.Genre.unique()
generos

This tells us that we have 12 distinct video game genres. 

###?We can now generate a label encoding scheme for mapping each category to a numeric value by leveraging scikit-learn LabelEncoder

In [ ]:
# usando LabelEncoder muestra los g?neros y las categor?as asociadas a cada g?nero
from sklearn import preprocessing

le = preprocessing.LabelEncoder()
le.fit(generos)

le.classes_


In [ ]:
le.transform(le.classes_)

### Show the transformed labels values and the dataframe

In [ ]:
# primero muestra solo los g?neros del DataFrame
vg_df.Genre

In [ ]:
# muestra en el DataFrame los g?neros y sus categor?as asociadas 
vg_df['Genero_OneHot_Encoder'] = le.transform(vg_df.Genre)
vg_df.loc[:, ['Genre', 'Genero_OneHot_Encoder']].head(10)



# Transforming Ordinal Features

Ordinal attributes are categorical attributes with a sense of order amongst the values. Let?s consider the Pok?mon dataset. Let?s focus more specifically on the Type 1 attribute. We will think that each Type 1 has a different power that we can order.


In [ ]:
# lee Pokemon.csv y muestra un head()
poke_df = pd.read_csv("Ficheros/Pokemon.csv", encoding = 'latin1', index_col = 0)
poke_df.head()

In [ ]:
# usa un sample() con semilla 1 y toma todo el DataFrame para desordenarlo aleatoriamente
# resetea los ?ndices y haz un head()

poke_df = poke_df.sample(random_state = 1, frac = 1).reset_index(drop=True)
poke_df.head()



In [ ]:
# muestra las columnas del DataFrame
poke_df.columns

### Show the different type 1 present in the dataset

In general, there is no generic module or function to map and transform these features into numeric representations based on order automatically. Hence we can use a custom encoding\mapping scheme based on a dictionary.

In [ ]:
# escribe un diccionario que mapee el Type 1 con un n?mero asociado a c?mo es de bueno el Type 1.
# Es decir, presup?n que se pueden ordenar esas etiquetas.
# Usa DataFrame['Type 1'].unique() para seleccionar esos valores en ese orden y asignarles 1,2,3...
# Por ejemplo: 'Bug' se corresponde con 1, 'Water' se corresponde con 2...

# poke_df['Type 1'].unique()

type_1_map = {'Bug':1, 'Water':2, 'Rock':3, 'Normal':4, 'Fighting':5, 'Grass':6, 'Poison':7,
       'Fire':8, 'Ghost':9, 'Fairy':10, 'Electric':11, 'Dragon':12, 'Ground':13,
       'Psychic':14, 'Ice':15}


# mapea los valores en el DataFrame en una columna que se llame 'type_1_num'
# haz un head()

poke_df['type_1_num'] = poke_df['Type 1'].map(type_1_map)
poke_df.head()


# Encoding Categorical Features

## One-hot Encoding Scheme

In [ ]:
poke_df[['Name', 'Stage', 'Legendary']].iloc[4:10]

In [ ]:
# usa LabelEncoder

from sklearn.preprocessing import LabelEncoder

# transform and map pokemon Type 1 with LabelEncoder
# el m?todo zip te puede ayudar
le_pokemon = preprocessing.LabelEncoder()
le_pokemon.fit(poke_df['Type 1'])

le_pokemon.classes_
le_pokemon.transform(le_pokemon.classes_)

poke_df['Type 1 zip'] = poke_df['Type 1'].map(dict(zip(le_pokemon.classes_, le_pokemon.transform(le_pokemon.classes_))))


# transform and map pokemon legendary status with Label Encoder



In [ ]:
# Otra forma m?s sencilla utilizando transform
# ?Para esto vale fit y transform!
# Muchas transformaciones se dividen en fit (ajusta los par?metros de la transformaci?n)
# y en transform (aplica los cambios)

le_pokemon.fit(poke_df['Type 1'])
poke_df['Type 1 transformed'] = le_pokemon.transform(poke_df['Type 1'])

In [ ]:
#haz un head()
poke_df.head()

In [ ]:
# comprobamos que la codificaci?n del m?todo es alfab?tica
poke_df.sort_values('Type 1 zip')['Type 1'].unique()

In [ ]:
# haz un head()
poke_df.head()

The features Type 1 zip and Legendary_zip now depict the numeric representations of our categorical features. Let?s now apply the one-hot encoding scheme on these features. Apply the get_dummies() method.

In [ ]:
# encode Type 1 labels using one-hot encoding scheme
one_hot_df_type_1 = pd.get_dummies(poke_df['Type 1'], prefix = 'Type_1')

# encode legendary status labels using one-hot encoding scheme
one_hot_df_legendary = pd.get_dummies(poke_df['Legendary'], prefix = 'Legendary')


In [ ]:

# compruebo que solo hay 4 pokemon legendarios
sum(one_hot_df_legendary['Legendary_True'])

In [ ]:
# concatena el DataFrame original con la codificaci?n de Type 1 y de Legendary 

df_one_hot = pd.concat([poke_df, one_hot_df_type_1, one_hot_df_legendary], axis=1)
df_one_hot


Consider you built this encoding scheme on your training data and built some model and now you have some new data which has to be engineered for features before predictions as follows.

In [ ]:


new_poke_df = pd.DataFrame([['PikaZoom', 'Bug', True], 
                           ['CharMyToast', 'Water', False]],
                           columns=['Name', 'Type 1', 'Legendary'])
new_poke_df


In [ ]:
# usando fit() y transform(), a?ade Type1_Label y Lgnd_Label en el DataFrame

le_pokemon.fit(poke_df['Type 1'])
new_type1_labels = le_pokemon.transform(new_poke_df['Type 1'])
new_poke_df['Type1_Label'] = new_type1_labels
le_pokemon.fit(poke_df['Legendary'])
new_leg_labels = le_pokemon.transform(new_poke_df['Legendary'])
new_poke_df['Lgnd_Label'] = new_leg_labels

new_poke_df[['Name', 'Type 1', 'Type1_Label', 'Legendary', 'Lgnd_Label']]

You can leverage scikit-learn?s excellent API here by calling the transform(?) function of the previously build LabeLEncoder objects on the new data.

## Dummy Coding Scheme

Let?s try applying dummy coding scheme on Pok?mon Type 1 by dropping the first level binary encoded feature (Type 1 = Bug).


In [ ]:
# haz un get_dummies para una codificaci?n dummy
# muestra las filas desde la 4 hasta la 9 (incluida)
dummy_features = pd.get_dummies(poke_df, columns = ['Type 1'], drop_first = True)
dummy_features.iloc[4:10]



If you want, you can also choose to drop the last level binary encoded feature

In [ ]:
# haz un fit() de Type 1 y mira las clases que aparecen
le_pokemon.fit(poke_df['Type 1'])
le_pokemon.classes_


In [ ]:
# haz un dummies sin eliminar ninguna columna que se obtenga solo de Type 1
# haz un head()

dummy_features_todas = pd.get_dummies(poke_df, columns=['Type 1'], prefix = ['Tipo1'] )
dummy_features_todas.head()



In [ ]:
# comprueba en la codificaci?n con la columna eliminada (dummy)
# pista: isin te puede ayudar
dummy_features_todas.columns.isin(['Tipo1_Bug'])


In [ ]:
# comprueba qu? hace el signo ~
~dummy_features_todas.columns.isin(['Tipo1_Bug'])


In [ ]:
# haz una lectura en el DataFrame con la ?ltima sentencia que emplea ~
dummy_features_todas.columns[~dummy_features_todas.columns.isin(['Tipo1_Bug'])]


In [ ]:
# as?gnalo a una variable y muestra un head
dummy_features_sin_primera = dummy_features_todas[dummy_features_todas.columns[~dummy_features_todas.columns.isin(['Tipo1_Bug'])]]
dummy_features_sin_primera.head()


## Feature Hashing scheme

Find the number of different 'Genre' in the dataset.

In [ ]:
# Usa vgsales.csv, l?elo y haz un head()

df_videojuegos = pd.read_csv('Ficheros/vgsales.csv')
df_videojuegos.head()


In [ ]:
print('Total game genres: ' + str(len(df_videojuegos.Genre.unique())))
print(df_videojuegos.Genre.sort_values().unique())

### We can see that there are a total of 12 genres of video games. If we used a one-hot encoding scheme on the Genre feature, we would end up having 12 binary features. Instead, we will now use a feature hashing scheme by leveraging scikit-learn?s FeatureHasher class, which uses a signed 32-bit version of the Murmurhash3 hash function. We will pre-define the final feature vector size to be 6 in this case.

In [ ]:
from sklearn.feature_extraction import FeatureHasher
h = FeatureHasher(n_features = 6, input_type = 'string')
f = h.transform(df_videojuegos['Genre'])

f.toarray()

In [ ]:
f.toarray().shape

In [ ]:
df_videojuegos.Genre.size

In [ ]:
# 1 - hacer un DataFrame
# 2 - que tenga 2 columnas con categor?as
# 3 - hacer un get_dummies con drop_first=True de esas 2 columnas
# 4 - hacer un get_dummies con drop_first=False de esas 2 columnas

In [ ]:
df = pd.DataFrame({'City': ['Madrid', 'Madrid', 'NYC', 'NYC', 'NYC', 'Toronto', 'Toronto', 'Oslo', 'Oslo'],
                   'Language': ['Spanish', 'Spanish', 'English', 'English', 'English', 'French', 'French', 'Norwegian', 'Norwegian'],
                  'Country': ['Spain', 'Spain', 'USA', 'USA', 'USA', 'Canada', 'Canada', 'Norway', 'Norway']})

In [ ]:
pd.get_dummies(df, columns=['City', 'Country'], prefix=['ciudad', 'pais'], drop_first=False)
pd.get_dummies(df, columns=['City', 'Country'], prefix=['ciudad', 'pais'], drop_first=True) # por orden alfab?tico